# Lab 3 Track B: Retention Offer Generation (demo)

Generates retention offers for at-risk customers with RAG over NorthStar's policy corpus (`offer_rag.py`).

**Pipeline:** policy docs → section chunks → Titan Embeddings v2 → FAISS (in memory) → three retrieval passes → Claude Haiku 4.5 draft → verification pass → 3-sentence offer.

**Before running:**
- AWS credentials with Bedrock access to `amazon.titan-embed-text-v2:0` and `us.anthropic.claude-haiku-4-5-20251001-v1:0` (see `docs/bedrock-access-verification.txt`)
- `pip install -r requirements-lab3.txt`
- Run from the repository root or from `notebooks/`

Nothing here needs the Lab 2 AWS stack: the model-flagged customers are read from `data/lab3_high_risk_customers.json`. A full run costs about $0.05 in Bedrock calls.

In [1]:
import os, sys
# Make the repo root importable whether the notebook runs from notebooks/ or the root.
root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, root)

from offer_rag import OfferRAG, describe_customer

rag = OfferRAG()   # defaults: strategy ABD, k=5, temperature 0, verification on
print(f"Indexed {len(rag.chunks)} policy chunks from northstar-policy-docs/")

Indexed 44 policy chunks from northstar-policy-docs/


## Customers the churn model flags as high-risk

`select_high_risk_customers.py` scored all 9,999 customers with the Task 1 model, flagged its top 10% (the same operating point as precision/recall@10%), and saved the highest-scoring flagged customer in each loyalty tier plus the next highest overall. Their top categories come from the cleaned Lab 2 transactions, observation window only. The file is committed so this notebook runs after the AWS stack is torn down.

In [2]:
import json

with open(os.path.join(root, "data", "lab3_high_risk_customers.json")) as f:
    selection = json.load(f)
print(f"Model flagged {selection['n_flagged']:,} of {selection['n_scored']:,} customers "
      f"({selection['flag_rule']}); model AUC {selection['model']['auc_roc']}")

customers = [(c["customer_id"], c["customer_context"], c["question"])
             for c in selection["customers"]]

Model flagged 1,000 of 9,999 customers (top 10% of scores (>= 0.6523)); model AUC 0.784


In [3]:
for customer_id, profile, question in customers:
    out = rag.generate(profile, question)
    print("=" * 100)
    print(f"{customer_id}  |  churn {profile['churn_probability']:.0%}  |  {profile['loyalty_tier']}  |  {out['latency_s']}s")
    print("\nINPUT")
    print(describe_customer(profile))
    print("\nRETRIEVED:", "; ".join(out["sections"]))
    print("\nOFFER")
    print(out["answer"])
    print()

CUST-10005299  |  churn 99%  |  Bronze  |  2.81s

INPUT
- Loyalty tier (analytics estimate): Bronze
- Churn probability: 99%
- Days since last purchase: 334.0
- Lifetime value: $141.75; average order $141.75
- Buys in 1 of 8 categories (a narrow shopper; cross-category discovery is an option)
- Channel: shops mostly in store, so shipping benefits matter little (online ratio 0.00)
- Top categories: Electronics
- Recommended category for this offer: Electronics (the offer recommends a category the customer already buys)

RETRIEVED: POL-LOY-011 4. Tier Benefits; KB-FAQ-022 Loyalty; POL-RET-004 1. Standard Return Window > 1.1 Extended Window for Platinum Members; POL-LOY-011 3. Points; POL-LOY-011 2. Tiers; POL-LOY-011 5. Promotional Offers > 5.1 Retention Offers; POL-LOY-011 5. Promotional Offers; POL-RET-004 4. Non-Returnable Merchandise; KB-FAQ-022 Returns; KB-FAQ-022 Guidance for Automated Agents; KB-FAQ-022 Shipping; POL-SHP-007 2. Free Shipping Thresholds

OFFER
Retention offer for t

CUST-10002372  |  churn 99%  |  Bronze  |  2.83s

INPUT
- Loyalty tier (analytics estimate): Bronze
- Churn probability: 99%
- Days since last purchase: 317.0
- Lifetime value: $245.51; average order $122.75
- Buys in 2 of 8 categories (a narrow shopper; cross-category discovery is an option)
- Channel: shops mostly in store, so shipping benefits matter little (online ratio 0.00)
- Top categories: Beauty, Footwear
- Recommended category for this offer: Beauty and Footwear (the offer recommends a category the customer already buys)

RETRIEVED: POL-LOY-011 4. Tier Benefits; KB-FAQ-022 Loyalty; POL-RET-004 1. Standard Return Window > 1.1 Extended Window for Platinum Members; POL-LOY-011 3. Points; POL-LOY-011 2. Tiers; POL-LOY-011 5. Promotional Offers > 5.1 Retention Offers; POL-LOY-011 5. Promotional Offers; POL-RET-004 4. Non-Returnable Merchandise; KB-FAQ-022 Returns; KB-FAQ-022 Guidance for Automated Agents; KB-FAQ-022 Shipping; POL-SHP-007 2. Free Shipping Thresholds

OFFER
Retentio

CUST-10005333  |  churn 99%  |  Silver  |  4.06s

INPUT
- Loyalty tier (analytics estimate): Silver
- Churn probability: 99%
- Days since last purchase: 362.0
- Lifetime value: $502.88; average order $502.88
- Buys in 1 of 8 categories (a narrow shopper; cross-category discovery is an option)
- Channel: shops mostly in store, so shipping benefits matter little (online ratio 0.00)
- Top categories: Beauty
- Recommended category for this offer: Beauty (the offer recommends a category the customer already buys)
- Tier caution: lifetime value is within 10% of the Silver threshold ($500). Official tier uses trailing 12-month spend, so do not state or imply benefits of any tier other than Silver.

RETRIEVED: POL-LOY-011 4. Tier Benefits; KB-FAQ-022 Loyalty; POL-LOY-011 2. Tiers; POL-RET-004 1. Standard Return Window > 1.1 Extended Window for Platinum Members; POL-LOY-011 3. Points; POL-LOY-011 5. Promotional Offers > 5.1 Retention Offers; POL-LOY-011 5. Promotional Offers; POL-RET-004 4. Non

CUST-10007187  |  churn 98%  |  Gold  |  3.17s

INPUT
- Loyalty tier (analytics estimate): Gold
- Churn probability: 98%
- Days since last purchase: 222.0
- Lifetime value: $2,051.33; average order $227.93
- Buys in 3 of 8 categories
- Channel: shops mostly in store, so shipping benefits matter little (online ratio 0.11)
- Top categories: Home, Footwear
- Recommended category for this offer: Home and Footwear (the offer recommends a category the customer already buys)
- Tier caution: lifetime value is within 10% of the Gold threshold ($2,000). Official tier uses trailing 12-month spend, so do not state or imply benefits of any tier other than Gold.

RETRIEVED: POL-LOY-011 4. Tier Benefits; KB-FAQ-022 Loyalty; POL-LOY-011 2. Tiers > 2.3 Tier Protection; POL-RET-004 1. Standard Return Window > 1.1 Extended Window for Platinum Members; POL-LOY-011 2. Tiers; POL-LOY-011 5. Promotional Offers > 5.1 Retention Offers; POL-LOY-011 5. Promotional Offers; POL-RET-004 4. Non-Returnable Merchandis

CUST-10002552  |  churn 98%  |  Platinum  |  3.15s

INPUT
- Loyalty tier (analytics estimate): Platinum
- Churn probability: 98%
- Days since last purchase: 199.0
- Lifetime value: $8,862.85; average order $443.14
- Buys in 5 of 8 categories
- Channel: shops mostly in store, so shipping benefits matter little (online ratio 0.20)
- Top categories: Toys, Electronics
- Recommended category for this offer: Toys and Electronics (the offer recommends a category the customer already buys)

RETRIEVED: POL-LOY-011 4. Tier Benefits; KB-FAQ-022 Loyalty; POL-RET-004 1. Standard Return Window > 1.1 Extended Window for Platinum Members; POL-LOY-011 2. Tiers > 2.3 Tier Protection; POL-LOY-011 2. Tiers; POL-LOY-011 5. Promotional Offers > 5.1 Retention Offers; POL-LOY-011 5. Promotional Offers; POL-RET-004 4. Non-Returnable Merchandise; KB-FAQ-022 Returns; KB-FAQ-022 Guidance for Automated Agents; KB-FAQ-022 Shipping; POL-SHP-007 2. Free Shipping Thresholds

OFFER
Retention offer for this at-risk Plat

In [4]:
u = rag.usage
print(f"Generation: {u['calls']} Bedrock calls (draft + verification per offer), "
      f"{u['input_tokens']:,} input / {u['output_tokens']:,} output tokens, ~${rag.cost_usd():.4f}")

Generation: 10 Bedrock calls (draft + verification per offer), 36,599 input / 902 output tokens, ~$0.0411


## Evaluation

RAGAS scoring (faithfulness, answer relevancy, context recall) runs through the provided harness, using a Bedrock judge:

```bash
python evaluation_harness.py --track B
```

Guardrail checks (prompt injection, system-prompt disclosure, out-of-scope requests, tier rules):

```bash
python tests/offer_guardrail_checks.py
```

Results and the design rationale are in `docs/lab3-model-design.md`.